# Fetch asset data for Similarity Pilot 2
- Cryptos: BTC, ETH, TRX (price data only)
- Stocks: CME, ICE, MSTR, PYPL, HOOD, SOFI, XYZ (price data + fundamentals)

Output structure under `similarity_pilot2/`:
- `cryptos/current/` + `cryptos/runs/run_YYYY-MM-DD/`
- `stocks/current/` + `stocks/runs/run_YYYY-MM-DD/`

In [1]:
import csv
import json
import os
import subprocess
import time
from datetime import datetime, timedelta

import yfinance as yf

/Users/paulgrass/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/paulgrass/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ======================== Config ========================

CRYPTOS = ["BTC-USD", "ETH-USD", "TRX-USD"]
STOCKS  = ["CME", "ICE", "MSTR", "PYPL", "HOOD", "SOFI", "XYZ"]

# GICS industry (6-digit gind) for each stock — used for P/B percentile peer group
INDUSTRY_MAP = {
    "CME":  "402030",  # Capital Markets
    "ICE":  "402030",  # Capital Markets
    "HOOD": "402030",  # Capital Markets
    "PYPL": "402010",  # Diversified Financial Services (Transaction Processing)
    "XYZ":  "402010",  # Diversified Financial Services (Transaction Processing)
    "SOFI": "402020",  # Consumer Finance
    "MSTR": "451030",  # IT Consulting & Other Services
}

# Sector labels for display
SECTOR_LABEL = {
    "CME": "Financials", "ICE": "Financials", "HOOD": "Financials",
    "PYPL": "Financials", "XYZ": "Financials", "SOFI": "Financials",
    "MSTR": "Technology",
}

END   = datetime.now()
START = END - timedelta(days=365)

REPO_ROOT = "/Users/paulgrass/Library/Mobile Documents/com~apple~CloudDocs/Documents/Programming/Git/pilot3-asset-data"
BASE_DIR  = os.path.join(REPO_ROOT, "similarity_pilot2")

# Compustat P/B data (GICS sectors 40+45, mktcap > $5B, P/B > 0)
WRDS_CSV = os.path.join(REPO_ROOT, "WRDS", "pb_ratio_sectors_40_45_with_industry.csv")
MKTCAP_FLOOR = 5000  # $5B in Compustat millions

SLEEP_SEC    = 15
MAX_RETRIES  = 3
RETRY_DELAY  = 15

In [3]:
# ======================== Helpers ========================

def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def copy_to_current(src_path, current_dir):
    os.makedirs(current_dir, exist_ok=True)
    dst = os.path.join(current_dir, os.path.basename(src_path))
    with open(src_path, "rb") as s, open(dst, "wb") as d:
        d.write(s.read())


def fetch_price_data(symbol, start, end, max_retries=MAX_RETRIES):
    """Download 365-day price history via yfinance. Returns list of [timestamp_ms, close]."""
    for attempt in range(1, max_retries + 1):
        try:
            tkr = yf.Ticker(symbol)
            df = tkr.history(start=start.strftime("%Y-%m-%d"),
                             end=end.strftime("%Y-%m-%d"),
                             auto_adjust=True)
            if df is None or df.empty:
                raise ValueError("Empty dataframe returned.")
            pts = [
                [int(ts.timestamp() * 1000), round(float(row["Close"]), 2)]
                for ts, row in df.iterrows()
                if row.get("Close") is not None
            ]
            if not pts:
                raise ValueError("No valid close prices.")
            return pts
        except Exception as e:
            print(f"   ⚠️  Attempt {attempt}/{max_retries} for {symbol}: {e}")
            if attempt < max_retries:
                time.sleep(RETRY_DELAY)
    return None


def fetch_fundamentals_yf(symbol):
    """Fetch market cap and dividend yield from yfinance."""
    try:
        tkr = yf.Ticker(symbol)
        info = tkr.info
        return {
            "marketcap_raw": info.get("marketCap"),
            "div_y_raw":     info.get("dividendYield"),
        }
    except Exception as e:
        print(f"   ⚠️  Fundamentals fetch failed for {symbol}: {e}")
        return None


def load_wrds_pb(csv_path, mktcap_floor=MKTCAP_FLOOR):
    """Load Compustat P/B data from CSV grouped by gind. Filters P/B > 0 and mktcap >= floor."""
    industry_pbs = {}
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pb = float(row["pb_ratio"])
            mktcap = float(row["mktcap"])
            if pb <= 0 or mktcap < mktcap_floor:
                continue
            gind = row["gind"]
            industry_pbs.setdefault(gind, []).append((row["tic"], pb))
    for gind in industry_pbs:
        industry_pbs[gind].sort(key=lambda x: x[1])
    return industry_pbs


def compute_pb_percentile(stock_ticker, gind, industry_pbs):
    """Compute percentile of stock within its GICS industry (excluding itself)."""
    peers = industry_pbs.get(gind, [])
    stock_pb = None
    peer_vals = []
    for tic, pb in peers:
        if tic == stock_ticker:
            stock_pb = pb
        else:
            peer_vals.append(pb)
    if stock_pb is None or not peer_vals:
        return None, None
    below = sum(1 for p in peer_vals if p < stock_pb)
    pctile = round(100 * below / len(peer_vals))
    return round(stock_pb, 2), pctile


def pb_to_valuation(pctile):
    if pctile is None:
        return None
    if pctile <= 30:
        return "Low"
    elif pctile >= 70:
        return "High"
    return "Mid"

In [4]:
# ======================== Setup directories ========================

date_str = datetime.now().strftime("%Y-%m-%d")

crypto_run_dir = os.path.join(BASE_DIR, "cryptos", "runs", f"run_{date_str}")
crypto_cur_dir = os.path.join(BASE_DIR, "cryptos", "current")
stock_run_dir  = os.path.join(BASE_DIR, "stocks", "runs", f"run_{date_str}")
stock_cur_dir  = os.path.join(BASE_DIR, "stocks", "current")

for d in [crypto_run_dir, crypto_cur_dir, stock_run_dir, stock_cur_dir]:
    os.makedirs(d, exist_ok=True)

errors = []

In [5]:
# ======================== 1. Crypto price data ========================

for sym in CRYPTOS:
    print(f"⏳ {sym}…")
    pts = fetch_price_data(sym, START, END)
    if pts:
        out_name = sym.replace("-USD", "").lower() + "_365d.json"
        out_path = os.path.join(crypto_run_dir, out_name)
        write_json(out_path, {"prices": pts})
        copy_to_current(out_path, crypto_cur_dir)
        print(f"  ✅ {out_name} ({len(pts)} points)")
    else:
        print(f"  ❌ Failed: {sym}")
        errors.append(sym)
    time.sleep(SLEEP_SEC)

⏳ BTC-USD…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ btc_365d.json (365 points)
⏳ ETH-USD…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ eth_365d.json (365 points)
⏳ TRX-USD…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ trx_365d.json (365 points)


In [6]:
# ======================== 2. Stock price data ========================

for sym in STOCKS:
    print(f"⏳ {sym}…")
    pts = fetch_price_data(sym, START, END)
    if pts:
        out_name = sym.lower() + "_365d.json"
        out_path = os.path.join(stock_run_dir, out_name)
        write_json(out_path, {"prices": pts})
        copy_to_current(out_path, stock_cur_dir)
        print(f"  ✅ {out_name} ({len(pts)} points)")
    else:
        print(f"  ❌ Failed: {sym}")
        errors.append(sym)
    time.sleep(SLEEP_SEC)

⏳ CME…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ cme_365d.json (251 points)
⏳ ICE…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ ice_365d.json (251 points)
⏳ MSTR…


...

  ✅ hood_365d.json (251 points)
⏳ SOFI…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ sofi_365d.json (251 points)
⏳ XYZ…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/history.py:173: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


  ✅ xyz_365d.json (251 points)


In [7]:
# ======================== 3a. Load Compustat P/B for industry percentiles ========================

print(f"📊 Loading Compustat P/B data from {WRDS_CSV} (mktcap >= ${MKTCAP_FLOOR}M)…")
industry_pbs = load_wrds_pb(WRDS_CSV)
for gind, pairs in sorted(industry_pbs.items()):
    if gind in INDUSTRY_MAP.values():
        print(f"  GICS {gind}: {len(pairs)} firms with P/B > 0")

📊 Loading Compustat P/B data from /Users/paulgrass/Library/Mobile Documents/com~apple~CloudDocs/Documents/Programming/Git/pilot3-asset-data/WRDS/pb_ratio_sectors_40_45_with_industry.csv (mktcap >= $5000M)…
  GICS 402010: 25 firms with P/B > 0
  GICS 402020: 11 firms with P/B > 0
  GICS 402030: 58 firms with P/B > 0
  GICS 451030: 72 firms with P/B > 0


In [8]:
# ======================== 3b. Fundamentals for target stocks ========================
# P/B + percentile from Compustat (by GICS industry); market cap + dividend yield from yfinance

fundamentals = {}

for sym in STOCKS:
    print(f"⏳ {sym}…")
    gind = INDUSTRY_MAP[sym]

    # P/B + percentile from Compustat
    pb, pb_pctile = compute_pb_percentile(sym, gind, industry_pbs)
    valuation = pb_to_valuation(pb_pctile)

    # Market cap + dividend yield from yfinance
    raw = fetch_fundamentals_yf(sym)
    if raw is not None:
        mc_raw = raw.get("marketcap_raw")
        div_raw = raw.get("div_y_raw")
        mc_millions = round(mc_raw / 1_000_000, 2) if mc_raw else None
        if div_raw is not None:
            div_pct = round(div_raw * 100, 2) if div_raw < 1 else round(div_raw, 2)
        else:
            div_pct = None
    else:
        mc_millions = None
        div_pct = None
        errors.append(f"{sym}_fundamentals_yf")

    sector_label = SECTOR_LABEL[sym]
    fundamentals[sym] = {
        "marketcap":         mc_millions,
        "pb_current":        pb,
        "pb_current_pctile": pb_pctile,
        "div_y":             div_pct,
        "valuation":         valuation,
        "sector":            sector_label,
    }
    print(f"  ✅ {sym}: mcap={mc_millions}M, P/B={pb}, pctile={pb_pctile}, "
          f"val={valuation}, div={div_pct}%, sector={sector_label}")
    time.sleep(SLEEP_SEC)

# Save
write_json(os.path.join(stock_run_dir, "fundamentals.json"), fundamentals)
write_json(os.path.join(stock_cur_dir, "fundamentals.json"), fundamentals)
print(f"\n✅ Saved fundamentals.json")
fundamentals

⏳ CME…


/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/paulgrass/anaconda3/lib/python3.11/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


  ✅ CME: mcap=109829.24M, P/B=3.41, pctile=54, val=Mid, div=1.71%, sector=Financials
⏳ ICE…
  ✅ ICE: mcap=90093.91M, P/B=3.18, pctile=51, val=Mid, div=1.32%, sector=Financials
⏳ MSTR…
  ✅ MSTR: mcap=45680.73M, P/B=1.07, pctile=1, val=Low, div=None%, sector=Technology
⏳ PYPL…
  ✅ PYPL: mcap=42820.11M, P/B=2.65, pctile=58, val=Mid, div=1.23%, sector=Financials
⏳ HOOD…
  ✅ HOOD: mcap=69319.25M, P/B=11.15, pctile=93, val=High, div=None%, sector=Financials
⏳ SOFI…
  ✅ SOFI: mcap=23324.45M, P/B=3.17, pctile=80, val=High, div=None%, sector=Financials
⏳ XYZ…
  ✅ XYZ: mcap=39072.86M, P/B=1.76, pctile=50, val=Mid, div=None%, sector=Financials

✅ Saved fundamentals.json


{'CME': {'marketcap': 109829.24,
  'pb_current': 3.41,
  'pb_current_pctile': 54,
  'div_y': 1.71,
  'valuation': 'Mid',
  'sector': 'Financials'},
 'ICE': {'marketcap': 90093.91,
  'pb_current': 3.18,
  'pb_current_pctile': 51,
  'div_y': 1.32,
  'valuation': 'Mid',
  'sector': 'Financials'},
 'MSTR': {'marketcap': 45680.73,
  'pb_current': 1.07,
  'pb_current_pctile': 1,
  'div_y': None,
  'valuation': 'Low',
  'sector': 'Technology'},
 'PYPL': {'marketcap': 42820.11,
  'pb_current': 2.65,
  'pb_current_pctile': 58,
  'div_y': 1.23,
  'valuation': 'Mid',
  'sector': 'Financials'},
 'HOOD': {'marketcap': 69319.25,
  'pb_current': 11.15,
  'pb_current_pctile': 93,
  'div_y': None,
  'valuation': 'High',
  'sector': 'Financials'},
 'SOFI': {'marketcap': 23324.45,
  'pb_current': 3.17,
  'pb_current_pctile': 80,
  'div_y': None,
  'valuation': 'High',
  'sector': 'Financials'},
 'XYZ': {'marketcap': 39072.86,
  'pb_current': 1.76,
  'pb_current_pctile': 50,
  'div_y': None,
  'valuation'

In [9]:
# ======================== 4. Save summary ========================

summary = {
    "date": date_str,
    "window_start": START.strftime("%Y-%m-%d"),
    "window_end": END.strftime("%Y-%m-%d"),
    "fundamentals": fundamentals,
    "errors": errors,
}

for d in [stock_run_dir, stock_cur_dir, crypto_run_dir, crypto_cur_dir]:
    write_json(os.path.join(d, "summary.json"), summary)

if errors:
    print(f"⚠️  {len(errors)} errors: {errors}")
else:
    print("✅ All symbols fetched successfully.")

✅ All symbols fetched successfully.


In [10]:
# ======================== 5. Git commit & push ========================
# Only run this cell if there are NO errors above!

assert not errors, f"Errors present — fix before pushing: {errors}"

os.chdir(REPO_ROOT)
!git add similarity_pilot2/
!git commit -m "similarity_pilot2: update data {date_str}"
!git push origin main

print("\nCDN base URLs (after push):")
print("  Cryptos: https://cdn.jsdelivr.net/gh/pagrass/pilot1-asset-data@latest/similarity_pilot2/cryptos/current/")
print("  Stocks:  https://cdn.jsdelivr.net/gh/pagrass/pilot1-asset-data@latest/similarity_pilot2/stocks/current/")

[main 106f108] similarity_pilot2: update data 2026-03-12
 27 files changed, 23268 insertions(+)
 create mode 100644 similarity_pilot2/.DS_Store
 create mode 100644 similarity_pilot2/cryptos/current/btc_365d.json
 create mode 100644 similarity_pilot2/cryptos/current/eth_365d.json
 create mode 100644 similarity_pilot2/cryptos/current/summary.json
 create mode 100644 similarity_pilot2/cryptos/current/trx_365d.json
 create mode 100644 similarity_pilot2/cryptos/runs/run_2026-03-12/btc_365d.json
 create mode 100644 similarity_pilot2/cryptos/runs/run_2026-03-12/eth_365d.json
 create mode 100644 similarity_pilot2/cryptos/runs/run_2026-03-12/summary.json
 create mode 100644 similarity_pilot2/cryptos/runs/run_2026-03-12/trx_365d.json
 create mode 100644 similarity_pilot2/stocks/current/cme_365d.json
 create mode 100644 similarity_pilot2/stocks/current/fundamentals.json
 create mode 100644 similarity_pilot2/stocks/current/hood_365d.json
 create mode 100644 similarity_pilot2/stocks/current/ice_365